This notebook (`MicroAblation_S4_S7_Validation.ipynb`) provides a focused validation study for two key architectural decisions within the S4 (QUR-RRF) and S7 (ComplianceGPT) retrieval pipelines.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === 1. IMPORTS & DEPENDENCIES ===

# Install required packages
!pip install -q pandas numpy torch faiss-cpu sentence-transformers rank_bm25

# Core libraries
import json
import math
import re
import os
import time
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional

# FAISS for dense indexing
import faiss

# Torch and Transformers
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

# Display options
pd.set_option("display.max_colwidth", 200)

print("All dependencies installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.5 MB/s eta 0:00:00
All dependencies installed and imported.


In [3]:
# === 2. CONFIGURATION ===
# All file paths needed for the retriever and the ablations

# --- 2a. Corpus & QUR Paths (Input) ---
# S1/S2 Corpus Paths (Used to build the indices)
REV5_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# S4/S7 QUR Paths (Used to get query rewrites)
QUR_CSV_REV5 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_error_bank.csv"

# --- 2b. Gold Sets (Input for Evaluation) ---
REV5_GOLD_CSV = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv"
ERROR_BANK_CSV = "/content/drive/MyDrive/compliance_data/error_bank/error_bank_v1.csv"

# --- 2c. Ablation Output Paths (Output) ---
# This is where all CSVs and reports from this notebook will be saved
ABLATION_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/micro_ablations"
os.makedirs(ABLATION_OUTPUT_DIR, exist_ok=True)
print(f"Ablation reports will be saved to: {ABLATION_OUTPUT_DIR}")

# --- 2d. Model & Search Parameters ---
BI_ENCODER_ID = 'intfloat/e5-small-v2'
CROSS_ENCODER_ID = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_N = 10              # Final metrics cutoff (e.g., MRR@10)
CANDIDATE_SET_SIZE = 50 # How many docs to retrieve in Stage 1 for the reranker
RRF_K = 60              # RRF fusion parameter

Ablation reports will be saved to: /content/drive/MyDrive/ablation_outputs/micro_ablations


In [4]:
# === 3. CORE LOGIC: TEXT PROCESSING & CORPUS LOADING ===
# These functions are copied from the S1 notebook to make this file self-contained.

PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")  # remove basic punctuation
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    """Lowercase, remove punctuation (keep letters/numbers/_), collapse whitespace."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    s = text.lower()
    s = s.replace('-', ' ') # Treat hyphens as spaces
    s = PUNCT_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

def tokenize(text: str) -> List[str]:
    """Alphanumeric tokens, lowercased (expects normalized text)."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    return [t.lower() for t in TOKEN_RE.findall(text)]

# ------------------------------
# Minimal BM25 (with precomputed TF)
# ------------------------------
class BM25Okapi:
    def __init__(self, corpus_tokens: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_tokens = corpus_tokens
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0

        # term freq per doc and doc freq
        self.tf: List[Dict[str, int]] = []
        df: Dict[str, int] = {}
        for doc in corpus_tokens:
            tf_doc: Dict[str, int] = {}
            for tok in doc:
                tf_doc[tok] = tf_doc.get(tok, 0) + 1
            self.tf.append(tf_doc)
            for term in tf_doc.keys():
                df[term] = df.get(term, 0) + 1

        # BM25+ style IDF (avoids negative idf)
        self.idf = {term: math.log((self.N - freq + 0.5) / (freq + 0.5) + 1.0) for term, freq in df.items()}

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        for term in query_tokens:
            idf_t = self.idf.get(term)
            if idf_t is None:
                continue
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(term, 0)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i] / (self.avgdl + 1e-9)))
                scores[i] += idf_t * (f * (self.k1 + 1)) / (denom + 1e-9)
        return scores

    def get_top_n(self, query_tokens: List[str], n: int = 10) -> List[int]:
        """Returns the indices of the top-n documents."""
        scores = self.get_scores(query_tokens)
        if n >= len(scores):
            return list(np.argsort(-scores))
        # Use argpartition for efficiency (faster than full sort)
        idx = np.argpartition(-scores, n-1)[:n]
        idx = idx[np.argsort(-scores[idx])]
        return list(idx)
# --- End of BM25 Class ---

def extract_text_from_record(rec: Dict) -> str:
    """Extracts all 'prose' fields from a NIST JSONL record."""
    texts: List[str] = []
    def recurse(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "prose" and isinstance(v, str):
                    texts.append(v)
                else:
                    recurse(v)
        elif isinstance(obj, list):
            for it in obj:
                recurse(it)
    # Add top-level title and prose
    if isinstance(rec.get("title"), str):
        texts.append(rec["title"])
    if isinstance(rec.get("prose"), str):
        texts.append(rec["prose"])
    # Recurse through parts and param guidelines
    recurse(rec.get("parts", []))
    for prm in rec.get("params") or []:
        if isinstance(prm, dict):
            recurse(prm.get("guidelines", []))
    return "\n".join(texts)

def extract_control_id(rec: Dict) -> Optional[str]:
    """Finds the best control ID (e.g., 'AC-3') from the props list."""
    candidates = []
    for prop in rec.get("props", []) or []:
        if isinstance(prop, dict) and prop.get("name") == "label":
            val = prop.get("value")
            if isinstance(val, str) and val.strip():
                candidates.append(val.strip())
    # Rank candidates: prefer parenthetical (e.g., "AC-1 (2)")
    # and non-zero-padded ("AC-3" > "AC-03")
    if candidates:
        def rank(v: str):
            has_paren = '(' in v
            zero_padded = bool(re.search(r"-0\d", v))
            return (has_paren, not zero_padded, len(v))
        candidates.sort(key=rank, reverse=True)
        return candidates[0]
    rid = rec.get("id")
    return rid.upper() if isinstance(rid, str) else None

def control_id_aliases(cid: str) -> List[str]:
    """Generates 'AC-3', 'AC 3', 'AC3'."""
    cid_clean = cid.strip()
    return [cid_clean, cid_clean.replace('-', ' '), cid_clean.replace('-', '')]

def load_corpus_jsonl(path: str) -> List[Dict[str, str]]:
    """Loads the NIST JSONL, returning a list of {'control_id': str, 'text': str} dicts."""
    docs = []
    print(f"  Loading corpus from: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            rec = json.loads(line)
            cid = extract_control_id(rec)
            if not cid: continue
            body = extract_text_from_record(rec)
            if not isinstance(body, str) or not body.strip(): continue
            # Prefix control_id variants to aid ID-based queries
            prefix = " ".join(control_id_aliases(cid))
            full_text = prefix + "\n" + body
            docs.append({"control_id": cid, "text": full_text})
    print(f"  Loaded {len(docs)} documents.")
    return docs

print("Core helper functions and BM25 class defined.")

Core helper functions and BM25 class defined.


In [5]:
# === 4. CORE LOGIC: METRICS & REPORTING ===
# Copied from S1 notebook for self-contained evaluation

def _to_int_rank(x) -> int:
    """Converts a rank (str, float, NaN) to a 0-based int. 0 means 'miss'."""
    try:
        if pd.isna(x): return 0
    except Exception: pass
    try:
        r = int(x)
        return r if r >= 0 else 0
    except Exception: return 0

def recall_at_k(ranks: List[int], k: int) -> float:
    """Calculates Recall@k. Ranks are 1-based (0 is a miss)."""
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    hits = sum(1 for r in ranks if 1 <= r <= k)
    return hits / len(ranks)

def mrr_at_k(ranks: List[int], k: int) -> float:
    """Calculates MRR@k. Ranks are 1-based (0 is a miss)."""
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    rrs = [(1.0 / r) if (1 <= r <= k) else 0.0 for r in ranks]
    return float(np.mean(rrs)) if rrs else 0.0

def ndcg_at_k(ranks: List[int], k: int) -> float:
    """Calculates nDCG@k. Ranks are 1-based (0 is a miss)."""
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    vals = [(1.0 / math.log2(r + 1.0)) if (1 <= r <= k) else 0.0 for r in ranks]
    return float(np.mean(vals)) if vals else 0.0

def _rank_list(df: pd.DataFrame) -> List[int]:
    """Helper to extract the 'rank' column for metrics functions."""
    if "rank" in df.columns:
        return [int(r) if pd.notna(r) else 0 for r in df["rank"].tolist()]
    else:
        print("Warning: 'rank' column not found in DataFrame.")
        return []

def _odp_mask(series: pd.Series) -> pd.Series:
    """Creates a boolean mask for rows that are ODP (not N/A, None, etc.)."""
    s = series.astype(str).str.strip()
    up = s.str.upper()
    return ~(s.eq("") | up.isin(["N/A", "NA", "NONE", "NULL", "NAN"]))

def display_summary_report(results_df: pd.DataFrame, version_name: str):
    """Prints a formatted markdown table of metrics for a given results DataFrame."""
    ranks_all = _rank_list(results_df)
    n_all = len(results_df)
    mask_odp = _odp_mask(results_df["odp_required"])
    odp_df = results_df[mask_odp].copy()
    ranks_odp = _rank_list(odp_df)
    n_odp = len(odp_df)

    # Compute metrics
    r1_all = recall_at_k(ranks_all, 1); r5_all = recall_at_k(ranks_all, 5); r10_all = recall_at_k(ranks_all, 10)
    mrr_all = mrr_at_k(ranks_all, 10); ndcg_all = ndcg_at_k(ranks_all, 10)
    r1_odp = recall_at_k(ranks_odp, 1); r5_odp = recall_at_k(ranks_odp, 5); r10_odp = recall_at_k(ranks_odp, 10)
    mrr_odp = mrr_at_k(ranks_odp, 10); ndcg_odp = ndcg_at_k(ranks_odp, 10)

    print(f"\n--- {version_name}: Performance ---")
    print(f"| Metric    | Overall ({n_all}q) | ODP-Subset ({n_odp}q) |")
    print("|-----------|------------------|--------------------|")
    print(f"| Recall@1  | {r1_all:.4f}           | {r1_odp:.4f}             |")
    print(f"| Recall@5  | {r5_all:.4f}           | {r5_odp:.4f}             |")
    print(f"| Recall@10 | {r10_all:.4f}           | {r10_odp:.4f}             |")
    print(f"| MRR@10    | {mrr_all:.4f}           | {mrr_odp:.4f}             |")
    print(f"| nDCG@10   | {ndcg_all:.4f}           | {ndcg_odp:.4f}             |")

def _compute_summary_values(results_df: pd.DataFrame):
    """Helper to compute all metrics and return them in a dict."""
    ranks_all = _rank_list(results_df)
    n_all = len(results_df)
    mask_odp = _odp_mask(results_df["odp_required"])
    odp_df = results_df[mask_odp].copy()
    ranks_odp = _rank_list(odp_df)
    n_odp = len(odp_df)
    return {
        "n_all": n_all, "n_odp": n_odp,
        "r1_all": recall_at_k(ranks_all, 1), "r5_all": recall_at_k(ranks_all, 5), "r10_all": recall_at_k(ranks_all, 10),
        "mrr_all": mrr_at_k(ranks_all, 10), "ndcg_all": ndcg_at_k(ranks_all, 10),
        "r1_odp": recall_at_k(ranks_odp, 1), "r5_odp": recall_at_k(ranks_odp, 5), "r10_odp": recall_at_k(ranks_odp, 10),
        "mrr_odp": mrr_at_k(ranks_odp, 10), "ndcg_odp": ndcg_at_k(ranks_odp, 10),
    }

def save_summary_report(rev5_df, rev4_df, out_path, system_name, err_rev5_df=None, err_rev4_df=None):
    """Saves a tab-separated metrics report to a .md file."""
    v5 = _compute_summary_values(rev5_df)
    v4 = _compute_summary_values(rev4_df)
    lines = [f"{system_name}: Performance Evaluation"]
    lines.append("\nNIST SP 800-53 Rev. 5: Performance")
    lines.append(f"Metric\tOverall ({v5['n_all']}q)\tODP-Subset ({v5['n_odp']}q)")
    lines.append(f"Recall@1\t{v5['r1_all']:.4f}\t{v5['r1_odp']:.4f}")
    lines.append(f"Recall@5\t{v5['r5_all']:.4f}\t{v5['r5_odp']:.4f}")
    lines.append(f"Recall@10\t{v5['r10_all']:.4f}\t{v5['r10_odp']:.4f}")
    lines.append(f"MRR@10\t{v5['mrr_all']:.4f}\t{v5['mrr_odp']:.4f}")
    lines.append(f"nDCG@10\t{v5['ndcg_all']:.4f}\t{v5['ndcg_odp']:.4f}")
    lines.append("\nNIST SP 800-53 Rev. 4: Performance")
    lines.append(f"Metric\tOverall ({v4['n_all']}q)\tODP-Subset ({v4['n_odp']}q)")
    lines.append(f"Recall@1\t{v4['r1_all']:.4f}\t{v4['r1_odp']:.4f}")
    lines.append(f"Recall@5\t{v4['r5_all']:.4f}\t{v4['r5_odp']:.4f}")
    lines.append(f"Recall@10\t{v4['r10_all']:.4f}\t{v4['r10_odp']:.4f}")
    lines.append(f"MRR@10\t{v4['mrr_all']:.4f}\t{v4['mrr_odp']:.4f}")
    lines.append(f"nDCG@10\t{v4['ndcg_all']:.4f}\t{v4['ndcg_odp']:.4f}")

    # Add Error Bank sections if provided
    if err_rev5_df is not None and not err_rev5_df.empty:
        v5_err = _compute_summary_values(err_rev5_df)
        lines.append(f"\nError Bank — Rev.5 only (n={v5_err['n_all']})")
        lines.append(f"Metric\tOverall ({v5_err['n_all']}q)\tODP-Subset ({v5_err['n_odp']}q)")
        lines.append(f"MRR@10\t{v5_err['mrr_all']:.4f}\t{v5_err['mrr_odp']:.4f}")

    if err_rev4_df is not None and not err_rev4_df.empty:
        v4_err = _compute_summary_values(err_rev4_df)
        lines.append(f"\nError Bank — Rev.4 only (n={v4_err['n_all']})")
        lines.append(f"Metric\tOverall ({v4_err['n_all']}q)\tODP-Subset ({v4_err['n_odp']}q)")
        lines.append(f"MRR@10\t{v4_err['mrr_all']:.4f}\t{v4_err['mrr_odp']:.4f}")

    text = "\n".join(lines)
    with open(out_path, "w", encoding="utf-8") as f: f.write(text)
    print(f"Saved summary report -> {os.path.abspath(out_path)}")

print("Metrics and reporting functions defined.")

Metrics and reporting functions defined.


In [6]:
# === 5. THE CORE CLASS: ComplianceGPTRetriever ===
# This class encapsulates all logic from S1, S2, S4, S6, and S7
# into a single, reusable object.

class ComplianceGPTRetriever:
    def __init__(self,
                 rev5_catalog_path: str,
                 rev4_catalog_path: str,
                 qur_rev5_path: str,
                 qur_rev4_path: str,
                 qur_errb_path: str,
                 device: Optional[str] = None):

        print("--- Initializing ComplianceGPT Retriever Engine ---")

        # 1. Set device (GPU if available)
        if device:
            self.device = device
        else:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # 2. Load Models (S2 Bi-Encoder, S6 Cross-Encoder)
        print(f"Loading Bi-Encoder ({BI_ENCODER_ID})...")
        self.bi_encoder = SentenceTransformer(BI_ENCODER_ID, device=self.device)
        self.bi_encoder_dim = self.bi_encoder.get_sentence_embedding_dimension()

        print(f"Loading Cross-Encoder ({CROSS_ENCODER_ID})...")
        self.cross_encoder = CrossEncoder(CROSS_ENCODER_ID, max_length=512, device=self.device)

        # 3. Build Indices (S1 BM25, S2 Dense)
        print("\nBuilding Rev. 5 Index...")
        self.rev5_bm25, self.rev5_dense, self.rev5_map, self.rev5_ids = self._build_index(rev5_catalog_path)
        print(f"Rev. 5 Index ready: {len(self.rev5_ids)} documents.")

        print("\nBuilding Rev. 4 Index...")
        self.rev4_bm25, self.rev4_dense, self.rev4_map, self.rev4_ids = self._build_index(rev4_catalog_path)
        print(f"Rev. 4 Index ready: {len(self.rev4_ids)} documents.")

        # 4. Load QUR Data (S4/S7)
        print("\nLoading QUR (Query Rewrite) data...")
        self.qur_rev5_df = self._load_qur_df(qur_rev5_path)
        self.qur_rev4_df = self._load_qur_df(qur_rev4_path)
        self.qur_errb_df = self._load_qur_df(qur_errb_path)
        print("QUR data loaded.")

        print("\n--- Retriever is Ready ---")

    def _load_qur_df(self, path: str) -> pd.DataFrame:
        """Loads and cleans a QUR CSV file."""
        df = pd.read_csv(path, encoding="ISO-8859-1")
        # Clean query strings for reliable matching
        df['original_query'] = df['original_query'].astype(str).str.strip().str.lstrip(": ")
        df['rewritten_query'] = df['rewritten_query'].astype(str).str.strip().str.lstrip(": ")
        return df

    def _build_index(self, catalog_path: str) -> Tuple:
        """Loads a corpus and builds S1 (BM25) and S2 (Dense) indices in memory."""
        # Load docs: [{'control_id': 'AC-1', 'text': '...'}, ...]
        docs = load_corpus_jsonl(catalog_path)

        # Create helper maps and lists
        id_to_text_map = {doc['control_id']: doc['text'] for doc in docs}
        control_ids = [doc['control_id'] for doc in docs]
        doc_texts = [doc['text'] for doc in docs] # Full text with ID prefix

        # --- Build S1: BM25 Index ---
        print("  Building BM25 index...")
        tokenized_corpus = [tokenize(normalize_text(t)) for t in doc_texts]
        bm25 = BM25Okapi(tokenized_corpus)

        # --- Build S2: Dense Index (FAISS) ---
        print(f"  Encoding {len(doc_texts)} passages for Dense index...")
        # e5 model expects "passage: " prefix for passages
        passages_to_encode = [f"passage: {t}" for t in doc_texts]

        X_embeddings = self.bi_encoder.encode(
            passages_to_encode,
            batch_size=128,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype('float32')

        print("  Building FAISS (IndexFlatIP) index...")
        # IndexFlatIP is for inner product (cosine similarity on normalized vectors)
        dense_index = faiss.IndexFlatIP(self.bi_encoder_dim)
        dense_index.add(X_embeddings)

        return bm25, dense_index, id_to_text_map, control_ids

    def _get_rewrites(self, original_query: str, qur_df: pd.DataFrame) -> Tuple[Optional[str], List[str]]:
        """Finds the best rewrite (rank 1) and all rewrites for a query."""
        q_cleaned = original_query.strip().lstrip(": ")
        matching_rows = qur_df[qur_df['original_query'] == q_cleaned]

        best_rewrite = None
        # Find the rank 1 rewrite
        best_rewrite_rows = matching_rows[matching_rows['rewrite_rank'] == 1]
        if not best_rewrite_rows.empty:
            best_rewrite = best_rewrite_rows['rewritten_query'].iloc[0]

        all_rewrites = matching_rows['rewritten_query'].tolist()

        return best_rewrite, all_rewrites

    def _find_rank(self, gold_cid: str, ranked_list: List[str], k: int) -> int:
        """Finds the 1-based rank of gold_cid in the list, or 0 if not in top k."""
        try:
            rank = ranked_list.index(gold_cid) + 1
            return rank if rank <= k else 0 # 0 means miss (not in top k)
        except ValueError:
            return 0 # 0 means miss (not found at all)

    def retrieve(self,
                 original_query: str,
                 gold_cid: str,
                 version: str = 'rev5',
                 mode: str = 's7_standard',
                 top_n: int = 10,
                 candidate_set_size: int = 50,
                 rrf_k: int = 60) -> int:
        """
        Main retrieval pipeline. Returns the 1-based rank of the gold_cid
        for the specified retrieval mode, or 0 if not found in top_n.

        Modes for Task - MicroAblation_S4_S7_Validation:
        - 's7_standard': Full S7 (QUR + Hybrid + Rerank(Original))
        - 'ablation_a':  Task (a). S7 but Rerank(BestRewrite)
        - 'ablation_b':  Task (b). S4-like RRF(Original + BestRewrite only)
        """

        # 1. Select the correct index components based on 'version'
        if version == 'rev5':
            bm25, dense, text_map, ids, qur_df = self.rev5_bm25, self.rev5_dense, self.rev5_map, self.rev5_ids, self.qur_rev5_df
        elif version == 'rev4':
            bm25, dense, text_map, ids, qur_df = self.rev4_bm25, self.rev4_dense, self.rev4_map, self.rev4_ids, self.qur_rev4_df
        else:
            # This logic handles the Error Bank, which has mixed rev5/rev4 queries
            q_cleaned = original_query.strip().lstrip(": ")
            is_rev5 = (self.qur_rev5_df['original_query'] == q_cleaned).any()
            if is_rev5:
                bm25, dense, text_map, ids, qur_df = self.rev5_bm25, self.rev5_dense, self.rev5_map, self.rev5_ids, self.qur_errb_df
            else:
                bm25, dense, text_map, ids, qur_df = self.rev4_bm25, self.rev4_dense, self.rev4_map, self.rev4_ids, self.qur_errb_df

        # 2. Get query rewrites (needed for all modes)
        best_rewrite, all_rewrites = self._get_rewrites(original_query, qur_df)

        # 3. --- RUN ABLATION (b) ---
        # S4 (QUR-RRF) using ONLY Original Query + Best Rewrite
        # This mode does NOT use the dense retriever or reranker.
        if mode == 'ablation_b':
            queries_to_run_bm25 = [original_query]
            if best_rewrite:
                queries_to_run_bm25.append(best_rewrite)
            queries_to_run_bm25 = list(dict.fromkeys(queries_to_run_bm25)) # de-dupe

            rrf_scores = {}
            for q in queries_to_run_bm25:
                q_tokens = tokenize(normalize_text(q))
                bm25_idxs = bm25.get_top_n(q_tokens, n=candidate_set_size)
                bm25_results = [ids[i] for i in bm25_idxs]
                # Apply RRF
                for rank, cid in enumerate(bm25_results, start=1):
                    rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))

            fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
            final_ranked_cids = [cid for cid, score in fused_ranking]
            return self._find_rank(gold_cid, final_ranked_cids, top_n)

        # 4. --- RUN S7 or ABLATION (a) ---
        # These modes both use the full S7 "Superhybrid" retrieval

        # --- Stage 1: "Superhybrid" RRF ---
        # This stage fuses BM25(Original + All Rewrites) + Dense(Original)
        rrf_scores = {}

        # Part 1A: S4-style (BM25 + QUR RRF)
        queries_to_run_bm25 = list(dict.fromkeys([original_query] + all_rewrites))
        for q in queries_to_run_bm25:
            q_tokens = tokenize(normalize_text(q))
            bm25_idxs = bm25.get_top_n(q_tokens, n=candidate_set_size)
            bm25_results = [ids[i] for i in bm25_idxs]
            for rank, cid in enumerate(bm25_results, start=1):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))

        # Part 1B: S2-style (Dense)
        # e5 model expects "query: " prefix for queries
        query_to_encode = [f"query: {original_query}"]
        q_embedding = self.bi_encoder.encode(
            query_to_encode,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype('float32')

        scores, idxs = dense.search(q_embedding, candidate_set_size)
        dense_results = [ids[i] for i in idxs[0] if i != -1]
        for rank, cid in enumerate(dense_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (rrf_k + rank))

        # Part 1C: RRF Fusion
        if not rrf_scores: return 0 # No results found
        fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
        candidate_cids = [cid for cid, score in fused_ranking[:candidate_set_size]]

        # --- Stage 2: Rerank with Cross-Encoder ---

        # This is the *only* difference between 's7_standard' and 'ablation_a'
        if mode == 'ablation_a':
            # Use the single best rewrite for reranking
            query_for_reranker = best_rewrite if best_rewrite else original_query
        else: # 's7_standard'
            # Use the original query for reranking
            query_for_reranker = original_query

        rerank_pairs = []
        rerank_cids_ordered = []
        for cid in candidate_cids:
            passage_text = text_map.get(cid)
            if passage_text:
                # Create (query, passage) pairs
                rerank_pairs.append((query_for_reranker, passage_text))
                rerank_cids_ordered.append(cid)

        if not rerank_pairs: return 0

        # Get scores from the cross-encoder
        rerank_scores = self.cross_encoder.predict(rerank_pairs, show_progress_bar=False)

        # Create a list of (cid, score) tuples and sort by score
        reranked_results = list(zip(rerank_cids_ordered, rerank_scores))
        reranked_results.sort(key=lambda item: item[1], reverse=True)
        final_ranked_cids = [cid for cid, score in reranked_results]

        # --- Stage 3: Find Rank ---
        return self._find_rank(gold_cid, final_ranked_cids, top_n)

print("ComplianceGPTRetriever class defined.")

ComplianceGPTRetriever class defined.


In [7]:
# === 6. EXECUTION BLOCK ===
# This cell runs the two micro-ablations as defined in the plan.

def run_ablation_evaluation(gold_df: pd.DataFrame,
                            engine: ComplianceGPTRetriever,
                            version_flag: str,
                            mode_flag: str,
                            top_n: int = 10,
                            candidate_set_size: int = 50,
                            rrf_k: int = 60) -> pd.DataFrame:
    """
    Helper function to loop over a gold set DataFrame and run the
    retriever engine for a specific mode.
    """
    records = []
    total = len(gold_df)
    start_time = time.time()

    for i, row in gold_df.iterrows():
        q = str(row.get("question", "")).strip()
        cid = str(row.get("control_id", "")).strip()
        odp = str(row.get("odp_required", "N/A")).strip()

        if not q or not cid: continue

        # Use the 'version' column for error bank, otherwise use the main flag
        version_override = row.get("version", version_flag).strip().lower()

        # Call the main retrieve method with the correct mode
        rank = engine.retrieve(
            original_query=q,
            gold_cid=cid,
            version=version_override,
            mode=mode_flag,
            top_n=top_n,
            candidate_set_size=candidate_set_size,
            rrf_k=rrf_k
        )

        records.append({
            "question": q,
            "gold_control_id": cid,
            "odp_required": odp,
            "rank": rank,
            "version": version_override
        })

        if (i + 1) % 25 == 0 or (i + 1) == total:
            print(f"  ... processed {i+1}/{total}")

    end_time = time.time()
    print(f"Evaluation finished in {end_time - start_time:.2f} seconds.")
    return pd.DataFrame(records)

# --- MAIN EXECUTION ---
try:
    # 1. Initialize the Engine (This will take a few minutes)
    print("--- Step 1: Initializing Retriever Engine ---")
    engine = ComplianceGPTRetriever(
        REV5_CATALOG_PATH, REV4_CATALOG_PATH,
        QUR_CSV_REV5, QUR_CSV_REV4, QUR_CSV_ERRB
    )

    # 2. Load all Gold Sets
    print("\n--- Step 2: Loading Gold Sets for Evaluation ---")
    rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1").dropna(subset=['question'])
    rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding="ISO-8859-1").dropna(subset=['question'])
    error_bank_gold = pd.read_csv(ERROR_BANK_CSV, encoding="ISO-8859-1").dropna(subset=['question'])

    # Split error bank
    error_bank_gold['version'] = error_bank_gold['version'].astype(str).str.lower().str.strip()
    err_rev5_gold = error_bank_gold[error_bank_gold['version'] == 'rev5'].copy()
    err_rev4_gold = error_bank_gold[error_bank_gold['version'] == 'rev4'].copy()
    print(f"Loaded {len(rev5_gold)} Rev5, {len(rev4_gold)} Rev4, and {len(error_bank_gold)} Error Bank queries.")

    # 3. --- Run Micro-Ablation (a) ---
    print("\n" + "="*50)
    print("🚀 RUNNING MICRO-ABLATION (a): S7 (Rerank vs. Best-Rewrite)")
    print("This re-runs S7, but passes (best_rewrite, passage) to the cross-encoder.")
    print("="*50)

    s7a_rev5_results = run_ablation_evaluation(rev5_gold, engine, 'rev5', 'ablation_a', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s7a_rev4_results = run_ablation_evaluation(rev4_gold, engine, 'rev4', 'ablation_a', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s7a_err5_results = run_ablation_evaluation(err_rev5_gold, engine, 'rev5', 'ablation_a', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s7a_err4_results = run_ablation_evaluation(err_rev4_gold, engine, 'rev4', 'ablation_a', TOP_N, CANDIDATE_SET_SIZE, RRF_K)

    print("\n--- Ablation (a) Results ---")
    display_summary_report(s7a_rev5_results, "Ablation (a) - Rev. 5")
    display_summary_report(s7a_rev4_results, "Ablation (a) - Rev. 4")
    display_summary_report(s7a_err5_results, "Ablation (a) - Error Bank (Rev. 5)")
    display_summary_report(s7a_err4_results, "Ablation (a) - Error Bank (Rev. 4)")

    save_summary_report(s7a_rev5_results, s7a_rev4_results,
                        os.path.join(ABLATION_OUTPUT_DIR, "S7a_RerankBest_Report.md"),
                        "Micro-Ablation (a): (S7a: S7 pipeline but with the Reranker using the Best-Rewrite) — Performance Evaluation",
                        s7a_err5_results, s7a_err4_results)

    # 4. --- Run Micro-Ablation (b) ---
    print("\n" + "="*50)
    print("RUNNING MICRO-ABLATION (b): S4 (RRF vs. Best-Rewrite)")
    print("This re-runs S4, but fuses ONLY (original_query, best_rewrite).")
    print("="*50)

    s4b_rev5_results = run_ablation_evaluation(rev5_gold, engine, 'rev5', 'ablation_b', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s4b_rev4_results = run_ablation_evaluation(rev4_gold, engine, 'rev4', 'ablation_b', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s4b_err5_results = run_ablation_evaluation(err_rev5_gold, engine, 'rev5', 'ablation_b', TOP_N, CANDIDATE_SET_SIZE, RRF_K)
    s4b_err4_results = run_ablation_evaluation(err_rev4_gold, engine, 'rev4', 'ablation_b', TOP_N, CANDIDATE_SET_SIZE, RRF_K)

    print("\n--- Ablation (b) Results ---")
    display_summary_report(s4b_rev5_results, "Ablation (b) - Rev. 5")
    display_summary_report(s4b_rev4_results, "Ablation (b) - Rev. 4")
    display_summary_report(s4b_err5_results, "Ablation (b) - Error Bank (Rev. 5)")
    display_summary_report(s4b_err4_results, "Ablation (b) - Error Bank (Rev. 4)")

    save_summary_report(s4b_rev5_results, s4b_rev4_results,
                        os.path.join(ABLATION_OUTPUT_DIR, "S4b_RRFBest_Report.md"),
                        "Micro-Ablation (b): (S4b: S4 pipeline but with the RRF using only the Best-Rewrite) — Performance Evaluation",
                        s4b_err5_results, s4b_err4_results)

    print("\n--- All Task Ablations Complete ---")

except Exception as e:
    print(f"\n--- CRITICAL ERROR ---")
    print(f"An unexpected error occurred: {e}")
    print("Please check all file paths in the 'Configuration' cell (Cell 2).")

--- Step 1: Initializing Retriever Engine ---
--- Initializing ComplianceGPT Retriever Engine ---
Using device: cuda
Loading Bi-Encoder (intfloat/e5-small-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...


Building Rev. 5 Index...
  Loading corpus from: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
  Loaded 1193 documents.
  Building BM25 index...
  Encoding 1193 passages for Dense index...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

  Building FAISS (IndexFlatIP) index...
Rev. 5 Index ready: 1193 documents.

Building Rev. 4 Index...
  Loading corpus from: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
  Loaded 922 documents.
  Building BM25 index...
  Encoding 922 passages for Dense index...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Building FAISS (IndexFlatIP) index...
Rev. 4 Index ready: 922 documents.

Loading QUR (Query Rewrite) data...
QUR data loaded.

--- Retriever is Ready ---

--- Step 2: Loading Gold Sets for Evaluation ---
Loaded 100 Rev5, 36 Rev4, and 34 Error Bank queries.

🚀 RUNNING MICRO-ABLATION (a): S7 (Rerank vs. Best-Rewrite)
This re-runs S7, but passes (best_rewrite, passage) to the cross-encoder.
  ... processed 25/100
  ... processed 50/100
  ... processed 75/100
  ... processed 100/100
Evaluation finished in 35.39 seconds.
  ... processed 25/36
  ... processed 36/36
Evaluation finished in 11.70 seconds.
  ... processed 23/23
Evaluation finished in 8.71 seconds.
  ... processed 25/11
Evaluation finished in 4.00 seconds.

--- Ablation (a) Results ---

--- Ablation (a) - Rev. 5: Performance ---
| Metric    | Overall (100q) | ODP-Subset (63q) |
|-----------|------------------|--------------------|
| Recall@1  | 0.7000           | 0.6667             |
| Recall@5  | 0.9500           | 0.9683    

In [8]:
# === 7. FINAL ANALYSIS & COMPARISON ===
# This cell creates the final comparison tables to prove the hypotheses
# and saves them to a central markdown report.

# --- Helper functions (from Cell 4) to make this cell self-contained ---
# This ensures: when re-run this cell, it has the functions it needs.
def _to_int_rank(x) -> int:
    try:
        if pd.isna(x): return 0
    except Exception: pass
    try:
        r = int(x)
        return r if r >= 0 else 0
    except Exception: return 0

def mrr_at_k(ranks: List[int], k: int) -> float:
    ranks = [_to_int_rank(r) for r in ranks]
    if not ranks: return 0.0
    rrs = [(1.0 / r) if (1 <= r <= k) else 0.0 for r in ranks]
    return float(np.mean(rrs)) if rrs else 0.0

def _rank_list(df: pd.DataFrame) -> List[int]:
    if "rank" in df.columns:
        return [int(r) if pd.notna(r) else 0 for r in df["rank"].tolist()]
    else:
        # Fallback for the summary dataframes (s7a_rev5_results, etc.)
        return [int(r) if pd.notna(r) else 0 for r in df["rank"].tolist()]
# --- End of helper functions ---


print("="*60)
print("Task (a) Analysis: S7 (Standard) vs. S7a (Ablation)")
print("Hypothesis: Reranking with the Original Query is SUPERIOR.")
print("="*60)

try:
    # S7 data (Baseline)
    # Cited from AblationStudy_S1_7.ipynb & S7_compliance_gpt_report.md
    s7_data = {
        "Dataset": ["Rev. 5 Overall", "Rev. 4 Overall", "Error Bank (Rev. 5)", "Error Bank (Rev. 4)"],
        "S7 (Standard) MRR@10": [0.9448, 0.9444, 0.8323, 0.8182],
    }

    # S7a data (Ablation)
    s7a_data = {
        "S7a (Rerank Best) MRR@10": [
            mrr_at_k(_rank_list(s7a_rev5_results), 10),
            mrr_at_k(_rank_list(s7a_rev4_results), 10),
            mrr_at_k(_rank_list(s7a_err5_results), 10),
            mrr_at_k(_rank_list(s7a_err4_results), 10)
        ]
    }

    s7_compare_df = pd.DataFrame(s7_data).set_index("Dataset")
    s7a_compare_df = pd.DataFrame(s7a_data, index=s7_data["Dataset"]).set_index(s7_compare_df.index)
    s7_final_df = s7_compare_df.join(s7a_compare_df)

    display(s7_final_df)

    print("\n\n" + "="*60)
    print("Task(b) Analysis: S4 (Standard) vs. S4b (Ablation)")
    print("Hypothesis: Fusing ALL rewrites is SUPERIOR.")
    print("="*60)

    # S4 data (Baseline)
    # Cited from AblationStudy_S1_7.ipynb
    s4_data = {
        "Dataset": ["Rev. 5 Overall", "Rev. 4 Overall", "Error Bank (Rev. 5)", "Error Bank (Rev. 4)"],
        "S4 (Standard) MRR@10": [0.7802, 0.8114, 0.4493, 0.5586],
    }

    # S4b data (Ablation)
    s4b_data = {
        "S4b (RRF Best) MRR@10": [
            mrr_at_k(_rank_list(s4b_rev5_results), 10),
            mrr_at_k(_rank_list(s4b_rev4_results), 10),
            mrr_at_k(_rank_list(s4b_err5_results), 10),
            mrr_at_k(_rank_list(s4b_err4_results), 10)
        ]
    }

    s4_compare_df = pd.DataFrame(s4_data).set_index("Dataset")
    s4b_compare_df = pd.DataFrame(s4b_data, index=s4_data["Dataset"]).set_index(s4_compare_df.index)
    s4_final_df = s4_compare_df.join(s4b_compare_df)

    display(s4_final_df)

    # --- Save tables to a single Markdown file ---
    md_content = []
    md_content.append("# Validation Study for S4 and S7 Retrieval Architectures: Comparison Report")
    md_content.append(f"Report generated: {pd.Timestamp.now()}")

    md_content.append("\n\n## Ablation (a): S7 (Standard) vs. S7a (Rerank Best)")
    md_content.append("**Hypothesis:** Reranking with the Original Query is SUPERIOR.")
    md_content.append("\n")
    md_content.append(s7_final_df.to_markdown())

    md_content.append("\n\n## Ablation (b): S4 (Standard) vs. S4b (RRF Best)")
    md_content.append("**Hypothesis:** Fusing ALL rewrites is SUPERIOR.")
    md_content.append("\n")
    md_content.append(s4_final_df.to_markdown())

    # ABLATION_OUTPUT_DIR is defined in Cell 2
    report_path = os.path.join(ABLATION_OUTPUT_DIR, "Comparison_Report-MicroAblation_S4_S7_Validation.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_content))

    print(f"\n\nSuccessfully saved comparison tables to:\n{report_path}")

except NameError as e:
    print(f"\n\n--- CRITICAL ERROR ---")
    print(f"NameError: {e}")
    print("This likely means the evaluation results (e.g., 's7a_rev5_results') are not in memory.")
    print("Please re-run 'Cell 6: EXECUTION BLOCK' and then run this cell again.")
except Exception as e:
    print(f"\n\nAn unexpected error occurred: {e}")

Task (a) Analysis: S7 (Standard) vs. S7a (Ablation)
Hypothesis: Reranking with the Original Query is SUPERIOR.


,S7 (Standard) MRR@10,S7a (Rerank Best) MRR@10
Dataset,,
Rev. 5 Overall,0.9448,0.817706
Rev. 4 Overall,0.9444,0.844907
Error Bank (Rev. 5),0.8323,0.698240
Error Bank (Rev. 4),0.8182,0.787879




Task(b) Analysis: S4 (Standard) vs. S4b (Ablation)
Hypothesis: Fusing ALL rewrites is SUPERIOR.


,S4 (Standard) MRR@10,S4b (RRF Best) MRR@10
Dataset,,
Rev. 5 Overall,0.7802,0.767429
Rev. 4 Overall,0.8114,0.778241
Error Bank (Rev. 5),0.4493,0.396791
Error Bank (Rev. 4),0.5586,0.583333




Successfully saved comparison tables to:
/content/drive/MyDrive/ablation_outputs/micro_ablations/Comparison_Report-MicroAblation_S4_S7_Validation.md
